In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


# =========================
# 0) 설정
# =========================
XLSX_PATH = r"/home/a202192020/맥주데이터실험/data/Supplemental Files and Figure source files.xlsx"
CHEM_SHEET = "Supplementary File S1"
SENS_SHEET = "Supplementary File S4"

OUT_ROOT = Path(r"/home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/origin/output")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

N_REPEATS = 10
TEST_SIZE = 0.30
STRATIFY_COL = "tasting_category_fine"
BASE_SEED = 20260227  # 반복 split의 seed 기준값(재현성 위해 고정)

STANDARDIZE_BEFORE_PCA = True
EIG_TOL = 1e-12

SAVE_PLOTS = True   # EVR 그래프 저장 여부


# =========================
# 1) 데이터 로드 + merge
# =========================
chem_df = pd.read_excel(XLSX_PATH, sheet_name=CHEM_SHEET)
sens_df = pd.read_excel(XLSX_PATH, sheet_name=SENS_SHEET)

META_COLS = ["beer", "beer_id", "tasting_category_fine"]
df = chem_df.merge(sens_df, on=META_COLS, how="inner", validate="one_to_one")

feature_cols = [c for c in chem_df.columns if c not in META_COLS]
target_cols  = [c for c in sens_df.columns if c not in META_COLS]

assert len(feature_cols) == 231
assert len(target_cols) == 50

print("data:", df.shape, "| #classes:", df[STRATIFY_COL].nunique())
print("min class count:", int(df[STRATIFY_COL].value_counts().min()))

# 소수 클래스 체크(현재 데이터는 min=3이어서 stratify split은 문제 없음)
# (만약 min < 2면 stratify가 불가능하므로 코드가 에러를 낼 수 있음)


# =========================
# 2) PCA 함수들
# =========================
def pca_fit_from_train(X_train: np.ndarray, standardize: bool, eig_tol: float):
    """
    train으로 PCA 축 생성 (공분산행렬 고유분해)
    return dict with mu,sigma,eigvals,eigvecs,keep_mask,V_keep,evr,evr_keep
    """
    Xtr = X_train.astype(float)
    mu = Xtr.mean(axis=0)
    Xtr_c = Xtr - mu

    if standardize:
        sigma = Xtr_c.std(axis=0, ddof=0)
        sigma_safe = sigma.copy()
        sigma_safe[sigma_safe == 0] = 1.0
        Xtr_cs = Xtr_c / sigma_safe
    else:
        sigma_safe = np.ones(Xtr.shape[1], dtype=float)
        Xtr_cs = Xtr_c

    n_train = Xtr_cs.shape[0]
    S = (Xtr_cs.T @ Xtr_cs) / (n_train - 1)

    eigvals, eigvecs = np.linalg.eigh(S)          # ascending
    idx = np.argsort(eigvals)[::-1]               # descending
    eigvals = np.clip(eigvals[idx], 0, None)
    eigvecs = eigvecs[:, idx]

    total_var = eigvals.sum()
    evr = eigvals / total_var if total_var > 0 else np.zeros_like(eigvals)

    keep_mask = eigvals > eig_tol
    V_keep = eigvecs[:, keep_mask]
    evr_keep = evr[keep_mask]

    return {
        "mu": mu,
        "sigma": sigma_safe,
        "eigvals": eigvals,
        "eigvecs": eigvecs,
        "evr": evr,
        "keep_mask": keep_mask,
        "V_keep": V_keep,
        "evr_keep": evr_keep,
    }

def pca_project(X: np.ndarray, mu: np.ndarray, sigma: np.ndarray, V_keep: np.ndarray, standardize: bool):
    Xc = X.astype(float) - mu
    Xcs = Xc / sigma if standardize else Xc
    Z = Xcs @ V_keep
    return Z

def pca_reconstruct(Z: np.ndarray, mu: np.ndarray, sigma: np.ndarray, V_keep: np.ndarray, standardize: bool):
    Xhat_cs = Z @ V_keep.T
    Xhat = Xhat_cs * sigma + mu if standardize else Xhat_cs + mu
    return Xhat

def recon_metrics(X_true: np.ndarray, X_hat: np.ndarray):
    err = X_true - X_hat
    return {
        "rmse": float(np.sqrt(np.mean(err**2))),
        "mae": float(np.mean(np.abs(err))),
        "max_abs_err": float(np.max(np.abs(err))),
    }

def worst_zscore_report(X: np.ndarray, mu: np.ndarray, sigma: np.ndarray, meta_df: pd.DataFrame, topn=10):
    """
    test에서 표준화 좌표가 크게 튀는 샘플/변수 추적용(복원 폭발 원인 찾기)
    """
    X = X.astype(float)
    Z = (X - mu) / sigma
    absmax_per_sample = np.max(np.abs(Z), axis=1)
    worst_idx = np.argsort(absmax_per_sample)[::-1][:topn]

    rows = []
    for i in worst_idx:
        j = int(np.argmax(np.abs(Z[i])))
        rows.append({
            "row_index_in_test": int(i),
            "beer": meta_df.iloc[i]["beer"],
            "beer_id": meta_df.iloc[i]["beer_id"],
            "style": meta_df.iloc[i][STRATIFY_COL],
            "max_abs_z": float(absmax_per_sample[i]),
            "worst_feature": feature_cols[j],
            "raw_value": float(X[i, j]),
            "mu": float(mu[j]),
            "sigma": float(sigma[j]),
            "z": float(Z[i, j]),
        })
    return pd.DataFrame(rows)


# =========================
# 3) 10번 반복 실행
# =========================
summary_rows = []

for rep in range(1, N_REPEATS + 1):
    run_dir = OUT_ROOT / str(rep)
    summary_dir = run_dir / "종합"
    summary_dir.mkdir(parents=True, exist_ok=True)

    seed = BASE_SEED + rep

    # ---- stratified split (논문처럼 70/30 + style 층화) ----
    train_df, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=seed,
        shuffle=True,
        stratify=df[STRATIFY_COL]
    )

    # split 저장(추적 가능하게)
    split_assign = pd.concat([
        train_df[META_COLS].assign(split="train", repeat=rep, random_state=seed),
        test_df[META_COLS].assign(split="test", repeat=rep, random_state=seed),
    ], axis=0).sort_values(["split", "tasting_category_fine", "beer_id"])

    split_path = summary_dir / "split_assignment.csv"
    split_assign.to_csv(split_path, index=False, encoding="utf-8-sig")

    # ---- X 만들기 + 결측치 처리(train 평균) ----
    X_train = train_df[feature_cols].copy()
    X_test  = test_df[feature_cols].copy()

    impute_means = X_train.mean(axis=0)
    X_train = X_train.fillna(impute_means)
    X_test  = X_test.fillna(impute_means)

    Xtr = X_train.values
    Xte = X_test.values

    # ---- PCA fit (train only) ----
    pca = pca_fit_from_train(Xtr, STANDARDIZE_BEFORE_PCA, EIG_TOL)
    mu, sigma = pca["mu"], pca["sigma"]
    eigvals, eigvecs, evr = pca["eigvals"], pca["eigvecs"], pca["evr"]
    keep_mask, V_keep, evr_keep = pca["keep_mask"], pca["V_keep"], pca["evr_keep"]
    k_keep = int(np.sum(keep_mask))

    # 축 저장(npz)
    np.savez(
        summary_dir / "pca_axes_full_231.npz",
        feature_cols=np.array(feature_cols, dtype=object),
        mu=mu, sigma=sigma,
        standardize_before_pca=np.array([STANDARDIZE_BEFORE_PCA]),
        eig_tol=np.array([EIG_TOL]),
        eigvals=eigvals, eigvecs=eigvecs, evr=evr,
    )
    np.savez(
        summary_dir / "pca_axes_kept_nonzero.npz",
        feature_cols=np.array(feature_cols, dtype=object),
        mu=mu, sigma=sigma,
        standardize_before_pca=np.array([STANDARDIZE_BEFORE_PCA]),
        eig_tol=np.array([EIG_TOL]),
        keep_mask=keep_mask,
        kept_full_indices=np.where(keep_mask)[0],
        V_keep=V_keep,
        evr_keep=evr_keep,
    )

    # EVR 저장
    ev_df = pd.DataFrame({
        "pc_index_1based_full": np.arange(1, len(eigvals)+1),
        "eigenvalue": eigvals,
        "explained_variance_ratio": evr,
        "explained_variance_ratio_percent": evr * 100,
        "kept_nonzero": keep_mask
    })
    ev_df.to_csv(summary_dir / "explained_variance_ratio_all_pcs.csv", index=False, encoding="utf-8-sig")

    if SAVE_PLOTS:
        plt.figure(figsize=(9,4))
        plt.plot(evr*100, marker="o", linewidth=1)
        plt.title("EVR per PC (%)")
        plt.xlabel("PC index")
        plt.ylabel("EVR (%)")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(summary_dir / "evr_per_pc.png", dpi=160)
        plt.close()

        plt.figure(figsize=(9,4))
        plt.plot(np.cumsum(evr)*100, marker="o", linewidth=1)
        plt.title("Cumulative EVR (%)")
        plt.xlabel("PC index")
        plt.ylabel("Cumulative EVR (%)")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(summary_dir / "cumulative_evr.png", dpi=160)
        plt.close()

    # ---- 투영/복원 + 복원력 저장 ----
    Ztr = pca_project(Xtr, mu, sigma, V_keep, STANDARDIZE_BEFORE_PCA)
    Zte = pca_project(Xte, mu, sigma, V_keep, STANDARDIZE_BEFORE_PCA)

    Xtr_hat = pca_reconstruct(Ztr, mu, sigma, V_keep, STANDARDIZE_BEFORE_PCA)
    Xte_hat = pca_reconstruct(Zte, mu, sigma, V_keep, STANDARDIZE_BEFORE_PCA)

    m_tr = recon_metrics(Xtr, Xtr_hat)
    m_te = recon_metrics(Xte, Xte_hat)

    recon = pd.DataFrame([{
        "repeat": rep,
        "random_state": seed,
        "n_train": Xtr.shape[0],
        "n_test": Xte.shape[0],
        "p": Xtr.shape[1],
        "k_keep": k_keep,
        "standardize_before_pca": STANDARDIZE_BEFORE_PCA,
        "eig_tol": EIG_TOL,
        "train_recon_rmse": m_tr["rmse"],
        "train_recon_mae": m_tr["mae"],
        "train_recon_max_abs_err": m_tr["max_abs_err"],
        "test_recon_rmse": m_te["rmse"],
        "test_recon_mae": m_te["mae"],
        "test_recon_max_abs_err": m_te["max_abs_err"],
    }])
    recon.to_csv(summary_dir / "reconstruction_check_train_test.csv", index=False, encoding="utf-8-sig")

    # ---- (강추) test에서 복원/투영을 터뜨리는 원인 후보(이상치) 저장 ----
    meta_test = test_df[META_COLS].reset_index(drop=True)
    outlier_df = worst_zscore_report(Xte, mu, sigma, meta_test, topn=10)
    outlier_df.to_csv(summary_dir / "test_outlier_report_top10.csv", index=False, encoding="utf-8-sig")

    summary_rows.append(recon.iloc[0].to_dict())

    print(f"[{rep}/{N_REPEATS}] saved -> {run_dir}")

# 반복 전체 요약 저장(OUT_ROOT 아래)
summary_all = pd.DataFrame(summary_rows)
summary_all.to_csv(OUT_ROOT / "summary_reconstruction_over_10_repeats.csv", index=False, encoding="utf-8-sig")
print("✅ saved:", OUT_ROOT / "summary_reconstruction_over_10_repeats.csv")

display(summary_all[[
    "repeat","k_keep",
    "train_recon_rmse","test_recon_rmse", 
    "test_recon_max_abs_err"
]].sort_values("test_recon_rmse", ascending=False).head(10))

data: (250, 284) | #classes: 22
min class count: 3
[1/10] saved -> /home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/origin/output/1
[2/10] saved -> /home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/origin/output/2
[3/10] saved -> /home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/origin/output/3
[4/10] saved -> /home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/origin/output/4
[5/10] saved -> /home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/origin/output/5
[6/10] saved -> /home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/origin/output/6
[7/10] saved -> /home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/origin/output/7
[8/10] saved -> /home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/origin/output/8
[9/10] saved -> /home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/origin/output/9
[10/10] saved -> /home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/origin/output/10
✅ saved: /home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/origin/output/summary_re

,repeat,k_keep,train_recon_rmse,test_recon_rmse,test_recon_max_abs_err
5,6,174,4.020131e-13,721.310333,94179.868493
2,3,174,1.519117e-13,641.739715,59967.927504
1,2,174,2.398211e-13,334.203928,42597.824512
4,5,174,2.268809e-13,332.674676,36118.749745
0,1,174,1.284973e-13,331.018773,40126.437298
9,10,174,1.022835e-13,203.563283,23513.952904
7,8,174,1.159744e-13,91.289222,8984.956001
3,4,174,2.605692e-13,25.475628,2267.152873
8,9,174,2.533386e-13,20.008271,1123.139595
6,7,174,2.294894e-13,14.841794,980.632864


In [1]:
print('hello')

hello
